# Pure Noise Experiment: Marchenko-Pastur Bulk

This notebook builds the pure-noise benchmark for the project.

We study a data matrix

$$
A \in \mathbb{R}^{T \times N}
$$

and its sample covariance matrix

$$
C = \frac{1}{T} A^T A.
$$

If the entries of A are independent, mean-zero noise with variance sigma squared, then in the high-dimensional regime

$$
N,T \to \infty, \qquad \frac{N}{T} \to q,
$$

the eigenvalues of C follow the Marchenko-Pastur law. The limiting support is

$$
\lambda_- = \sigma^2(1-\sqrt q)^2
$$

and

$$
\lambda_+ = \sigma^2(1+\sqrt q)^2.
$$

This experiment shows that even pure noise produces a nontrivial eigenvalue spectrum. This is the baseline we later compare against the one-factor spiked model, where the same sigma controls the idiosyncratic noise variance.


In [109]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")

## Helper Functions

Throughout this notebook we work with the centered sample covariance matrix. Given a raw data matrix A, we first subtract each column mean:

$$
X = A - \bar A.
$$

Then we compute

$$
C = \frac{1}{T} X^T X.
$$

Centering removes sample mean effects while preserving the variance scale sigma squared.

The Marchenko-Pastur density with noise variance sigma squared is

$$
\rho(\lambda)=\frac{1}{2\pi q \sigma^2 \lambda}\sqrt{(\lambda_+-\lambda)(\lambda-\lambda_-)}
$$

for lambda inside the interval.


In [111]:
def center_columns(A):
    """Subtract each column's sample mean."""
    return A - A.mean(axis=0, keepdims=True)


def sample_covariance(A, center=True):
    """Return centered C = X^T X / T for A in R^{T x N}."""
    X = center_columns(A) if center else A
    T = X.shape[0]
    return X.T @ X / T


def mp_edges(q, sigma2=1.0):
    """Marchenko-Pastur lower and upper edges."""
    low = sigma2 * (1 - np.sqrt(q)) ** 2
    high = sigma2 * (1 + np.sqrt(q)) ** 2
    return low, high


def mp_density(x, q, sigma2=1.0):
    """Marchenko-Pastur density on its continuous support."""
    low, high = mp_edges(q, sigma2=sigma2)
    density = np.zeros_like(x, dtype=float)
    mask = (x >= low) & (x <= high)
    density[mask] = np.sqrt((high - x[mask]) * (x[mask] - low)) / (2 * np.pi * q * sigma2 * x[mask])
    return density


## Experiment Setup

We choose

$$
T=3000, \qquad N=1000.
$$

Therefore

$$
q = \frac{N}{T} = \frac{1}{3}.
$$

For consistency with the spiked covariance experiment, we explicitly set the pure-noise variance to sigma squared:

$$
E_{t,i} \sim N(0,\sigma^2).
$$

In the numerical experiment below we use sigma = 1. Since the data are generated with population mean zero, centering only removes finite-sample mean fluctuations and does not change the asymptotic MP benchmark.


In [113]:
T = 3000
N = 1000
q = N / T
sigma = 1.0
sigma2 = sigma**2
rng = np.random.default_rng(42)

mp_low, mp_high = mp_edges(q, sigma2=sigma2)

print(f"T={T}, N={N}, q=N/T={q:.3f}")
print(f"sigma = {sigma:.3f}")
print(f"MP lower edge = {mp_low:.3f}")
print(f"MP upper edge = {mp_high:.3f}")


T=3000, N=1000, q=N/T=0.333
sigma = 1.000
MP lower edge = 0.179
MP upper edge = 2.488


## Experiment 1: Repeated Gaussian Pure Noise Samples

Instead of using only one pure-noise matrix, we repeat the null experiment many times.

For each trial, we generate

$$
A_{t,i} \sim N(0,\sigma^2),
$$

center the columns, compute the sample covariance matrix, and collect all eigenvalues.

We then plot the frequency histogram of all eigenvalues from all trials, and mark the MP bulk interval. This focuses on the bulk-support prediction rather than the detailed MP density curve.


In [ ]:
n_trials = 50

all_eigs = np.empty(n_trials * N)
inside_props = np.zeros(n_trials)
lambda_mins = np.zeros(n_trials)
lambda_maxs = np.zeros(n_trials)

for trial in range(n_trials):
    A_trial = rng.normal(0.0, sigma, size=(T, N))
    C_trial = sample_covariance(A_trial)
    eigs = np.linalg.eigvalsh(C_trial)

    start = trial * N
    all_eigs[start:start + N] = eigs

    inside = (eigs >= mp_low) & (eigs <= mp_high)
    inside_props[trial] = inside.mean()
    lambda_mins[trial] = eigs[0]
    lambda_maxs[trial] = eigs[-1]

freq_inside = ((all_eigs >= mp_low) & (all_eigs <= mp_high)).mean()
freq_below = (all_eigs < mp_low).mean()
freq_above = (all_eigs > mp_high).mean()

plt.figure(figsize=(11, 6))
plt.axvspan(mp_low, mp_high, color="cornflowerblue", alpha=0.14, label="MP bulk")
plt.hist(
    all_eigs,
    bins=80,
    density=False,
    color="cornflowerblue",
    alpha=0.78,
    edgecolor="black",
    linewidth=0.35,
    label=f"sample eigenvalues from {n_trials} trials",
)
plt.axvline(mp_low, color="red", linestyle="--", linewidth=2.0, label=f"MP lower edge = {mp_low:.3f}")
plt.axvline(mp_high, color="red", linestyle="--", linewidth=2.0, label=f"MP upper edge = {mp_high:.3f}")
plt.xlabel("sample covariance eigenvalue")
plt.ylabel("frequency")
plt.title(f"Repeated Pure Noise: Eigenvalue Frequency vs MP Bulk (T={T}, N={N}, trials={n_trials})")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig("Imperial_College_London_Poster_Template/Images/research_figures/pure_noise_bulk_histogram.pdf", bbox_inches="tight")
plt.savefig("Imperial_College_London_Poster_Template/Images/research_figures/pure_noise_bulk_histogram.png", dpi=220, bbox_inches="tight")
plt.show()

print(f"trials = {n_trials}")
print(f"total eigenvalues checked = {all_eigs.size}")
print(f"frequency below MP bulk  = {freq_below:.5f}")
print(f"frequency inside MP bulk = {freq_inside:.5f}")
print(f"frequency above MP bulk  = {freq_above:.5f}")
print(f"mean trial-level inside proportion = {inside_props.mean():.5f}")
print(f"mean smallest eigenvalue = {lambda_mins.mean():.3f}")
print(f"mean largest eigenvalue  = {lambda_maxs.mean():.3f}")


## Interpretation

This experiment uses the centered sample covariance matrix

$$
C = \frac{1}{T} X^T X
$$

where X is obtained by subtracting each column mean from A.

Across repeated pure-noise samples, we pool all sample covariance eigenvalues and compare their frequency histogram with the MP bulk interval:

$$
[\lambda_-,\lambda_+]
=\left[\sigma^2(1-\sqrt q)^2,\,\sigma^2(1+\sqrt q)^2\right].
$$

The important message is:

> Under the pure-noise null model, the repeated-sample eigenvalue histogram is essentially contained in the MP bulk.

Therefore, in the spiked covariance experiment, a separated eigenvalue above the MP upper edge is meaningful only relative to this repeated pure-noise benchmark.
